# 5G Network Intelligence — MLP and Hurdle Forecast Blending

## Question and negative evidence

Residual MLP beats matched Linear but loses to hurdle in 10 of 12 comparisons. Standalone MAE does not establish forecast-error complementarity, and no benefit is assumed. Test only a small predefined convex blend, retaining weight zero.

## Temporal protocol

Select variant and weight on inner1 only for each horizon. Save every choice before reading inner2 numerical forecasts/targets. Apply those choices unchanged on inner2. Do not optimize again after confirmation. Both folds are reused development evidence, not fresh unseen-test data. Validation 504–671, original final holdout and production are excluded.


In [ ]:
RUN_SELECTION = False
SOURCE_COMMIT = '14a20592de48ee1cf072b96cff9bf3b1fa1ea9f0'


## Fixed configuration

No base-model fitting. Choosing coefficients is calibration and runs only on hosted Kaggle. Weight grid: 0, 0.05, 0.1, 0.2. Ties favor smaller weight, then A. Confirmation requires no horizon regression and at least one strict gain with nonzero weight. Selecting only zeros is not an improvement.


In [ ]:
PROTOCOL = {'protocol': 'development-mlp-hurdle-blend-v1',
 'selection_fold': 'inner1',
 'confirmation_fold': 'inner2',
 'horizons': [1, 6, 24],
 'variants': ['A', 'B'],
 'mlp_weight_grid': [0.0, 0.05, 0.1, 0.2],
 'selection_metric': 'original_unit_MAE',
 'tie_policy': 'smaller_weight_then_A_before_B',
 'confirmation_rule': 'no_horizon_regresses_and_at_least_one_strictly_improves_with_nonzero_weight',
 'post_confirmation_weight_changes': 'forbidden',
 'base_model_retraining': False,
 'validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden',
 'mlp_manifest_sha256': 'e9dd565ee26362c3ec2f79ff5061722df2a72c83ca3d434944c0f9042b115ba1',
 'hurdle_manifest_sha256': '593d524646ed96e1b2c2d62eb67892f8698c7b23d08dff6a7420b3bbd918efb0',
 'claim_scope': 'reused_development_only_not_fresh_confirmation'}


## Prediction arithmetic

The following cell only defines the fixed-weight formula. It does not search coefficients or fit a model. Both forecasts must cover the same targets.


In [ ]:
"""Fixed-coefficient forecast arithmetic only; no fitting or selection."""

import numpy as np


def blend_forecasts(hurdle, linear, linear_weight):
    hurdle, linear = np.asarray(hurdle), np.asarray(linear)
    if hurdle.shape != linear.shape or not np.isfinite(linear_weight) or not 0 <= linear_weight <= 1:
        raise ValueError("Aligned forecasts and scalar weight in [0,1] required")
    if not np.isfinite(hurdle).all() or not np.isfinite(linear).all():
        raise ValueError("Finite forecasts required")
    # Preserve original stored precision: float64 mixing is declared for reproducibility.
    return (1-linear_weight)*hurdle.astype(float) + linear_weight*linear.astype(float)


## Completed source outputs

Attach the original hurdle development job and Residual MLP job. Their complete artifact manifests are verified before selection. Source, config and forecast provenance remain separate from fitted blend choices.


In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,tarfile
if RUN_SELECTION:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required for coefficient selection')
    hurdle_roots = [p.parent.parent for p in Path('/kaggle/input').rglob('hurdle_development_scores.csv') if (p.parent.parent/'artifacts/hurdle_research/manifest.json').exists()]
    mlp_roots = [p.parent.parent for p in Path('/kaggle/input').rglob('temporal_mlp_development.csv') if (p.parent.parent/'artifacts/temporal_mlp/manifest.json').exists()]
    if len(hurdle_roots)!=1 or len(mlp_roots)!=1:
        raise RuntimeError('Attach exactly one completed output for each base family')
    HURDLE = hurdle_roots[0]/'artifacts/hurdle_research'
    MLP = mlp_roots[0]/'artifacts/temporal_mlp'
    PROJECT = Path('/kaggle/working/mlp_blend')
    PROJECT.mkdir(exist_ok=True)
    shutil.copytree(hurdle_roots[0]/'src',PROJECT/'src',dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT']=SOURCE_COMMIT
    for folder in ('scripts','configs','reports'):
        Path(folder).mkdir(exist_ok=True)
    Path('scripts/__init__.py').touch()
    patches = list(Path('/kaggle/input').rglob('select_mlp_blend.py'))
    if len(patches)!=1:
        raise RuntimeError('Attach exactly one blend source dataset')
    patch = patches[0].parent
    for name,folder in (('select_development_blend.py','scripts'),('select_mlp_blend.py','scripts'),('forecast_blend.py','src/models'),('mlp_blend.yaml','configs')):
        shutil.copyfile(patch/name,Path(folder)/name)


## Selection, fixed confirmation and debugging

Checkpoints remain immutable. Save the full inner1 grid, selected coefficients, inner2 forecasts and a single pass/fail decision. No search on inner2 and no automatic follow-up validation job.


In [ ]:
if RUN_SELECTION:
    command=[sys.executable,'-u','-m','scripts.select_mlp_blend','--mlp-root',str(MLP),'--hurdle-root',str(HURDLE)]
    with Path('reports/mlp_blend.log').open('w') as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status=process.wait()
    with tarfile.open('/kaggle/working/mlp_blend_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/mlp_blend'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Blend experiment failed; inspect saved log')
else:
    print('Review mode: no selection, fitting or calibration.')


## Saved results and limitations

MAE is in original units; WAPE is a ratio. Report all horizons including regressions. Reused development confirmation does not establish generalization or absence of overfitting. Zero-weight selection must be reported as no gain.


In [ ]:
for name in ('mlp_blend_selection_grid.csv','mlp_blend_confirmation.csv'):
    paths=[Path('reports')/name,Path('../reports')/name]
    if Path('/kaggle/input').is_dir():
        paths+=list(Path('/kaggle/input').rglob(name))
    path=next((p for p in paths if p.exists()),None)
    if path:
        import pandas as pd
        from IPython.display import display
        print(name)
        display(pd.read_csv(path))
    else:
        print(f'{name}: no completed result attached')
